# multi

> Shared multi-agent event queue, training loop, and team environment.

Spec: `specs/multiagent/phase5/`. One loop (`runTeam`) runs a team of members in one world, always running the member whose next event starts earliest (ties go to the lower agent index). `TeamEnv` gives the same schedule one agent at a time, for actions chosen outside the package.

In [ ]:
#| default_exp multi

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import heapq
import json
from pathlib import Path
from typing import NamedTuple

import numpy as np

## Records and history

In [ ]:
#| export
class Record(NamedTuple):
    "One event of one agent."
    agent: int        # agent index
    t: object         # event start, in the team's time unit (int for 'sample', float for 's')
    dt: object        # event duration, same unit, > 0
    kind: str         # 'fixation' / 'saccade' (flexible), 'event' (pre-structured), 'iteration' (circuit), 'step', 'stay' / 'travel' (patch world)
    location: object  # where the reward was earned: the location at the end of the event
    reward: float
    extras: tuple     # the member's native row, in the order of member.extraNames


class Outcome(NamedTuple):
    "What `Member.event` returns; `runTeam` adds the agent index and start time."
    dt: object
    kind: str
    location: object
    reward: float
    extras: tuple = ()

In [ ]:
#| export
def _plain(x):
    "numpy scalars and arrays as plain Python values, for JSON."
    if isinstance(x, np.generic):
        return x.item()
    if isinstance(x, np.ndarray):
        return x.tolist()
    return x


class TeamHistory:
    "Records in the order they ran; `extraNames` holds one tuple of extras names per agent."
    def __init__(self, extraNames):
        self.extraNames = list(extraNames)
        self.records = []

    def add(self, record):
        self.records.append(record)

    def __len__(self):
        return len(self.records)

    def forAgent(self, i):
        "Records of agent i, in order."
        return [r for r in self.records if r.agent == i]

    def toJSONL(self, path):
        "One JSON object per record, extras keyed by name; returns the path."
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open('w', encoding='utf-8') as f:
            for r in self.records:
                names = self.extraNames[r.agent]
                f.write(json.dumps({
                    'agent': r.agent, 't': _plain(r.t), 'dt': _plain(r.dt), 'kind': r.kind,
                    'location': _plain(r.location), 'reward': _plain(r.reward),
                    'extras': {n: _plain(v) for n, v in zip(names, r.extras)},
                }) + '\n')
        return path


def spawnRngs(seed, n):
    "n independent generators from one seed."
    return [np.random.default_rng(s) for s in np.random.SeedSequence(seed).spawn(n)]

## Members and the team loop

Existing agents override `event` with their old loop body (see `members`). New agents for generic worlds write only `act` and `learn`; the default `event` calls observe, `act`, `apply`, `learn`. A generic world provides `reset(seed, nAgents)`, `observe(i)` and `apply(i, action) -> (reward, dt, location, info)`.

In [ ]:
#| export
class Member:
    "One agent in a team. `runTeam` / `TeamEnv` set `id` to the agent index."
    timeUnit = 's'
    extraNames = ()
    id = None

    def event(self, world, t) -> Outcome:
        "Default event for generic worlds: observe, act, apply, learn."
        obs = world.observe(self.id)
        action = self.act(obs)
        reward, dt, location, info = world.apply(self.id, action, t)
        self.learn(obs, action, reward, info)
        return Outcome(dt, 'step', location, reward, ())

    def act(self, obs):
        raise NotImplementedError

    def learn(self, obs, action, reward, info):
        pass

In [ ]:
#| export
class Clock:
    "Event queue of (next start time, agent index); members stop once their start time reaches t0 + Ttot."
    def __init__(self, n, Ttot, t0=0):
        self.tEnd = t0 + Ttot
        self.heap = [(t0, i) for i in range(n)]
        heapq.heapify(self.heap)

    def peek(self):
        "Index of the next member to run, or None when all are done."
        while self.heap and not self.heap[0][0] < self.tEnd:
            heapq.heappop(self.heap)
        return self.heap[0][1] if self.heap else None

    def pop(self):
        "(t, i) of the next member; call after `peek` returned i."
        return heapq.heappop(self.heap)

    def push(self, t, dt, i):
        "Reschedule member i at t + dt; True if it has more events."
        if not dt > 0:
            raise ValueError(f'event duration must be > 0, got {dt}')
        heapq.heappush(self.heap, (t + dt, i))
        return t + dt < self.tEnd


def runTeam(members, world, Ttot, t0=0) -> TeamHistory:
    "Run every member from t0 until t0 + Ttot; never resets the world or converts time."
    if not members:
        raise ValueError('team needs at least one member')
    units = {m.timeUnit for m in members}
    if len(units) != 1:
        raise ValueError(f'members must share one time unit, got {sorted(units)}')
    for i, m in enumerate(members):
        m.id = i
    clock = Clock(len(members), Ttot, t0)
    hist = TeamHistory([m.extraNames for m in members])
    while (i := clock.peek()) is not None:
        t, _ = clock.pop()
        out = members[i].event(world, t)
        hist.add(Record(i, t, out.dt, out.kind, out.location, out.reward, out.extras))
        clock.push(t, out.dt, i)
    return hist

## One agent at a time

In [ ]:
#| export
class TeamEnv:
    "Agent-environment-cycle view of a generic world: the caller picks each action."
    def __init__(self, world, nAgents, Ttot, t0=0):
        self.world, self.nAgents, self.Ttot, self.t0 = world, nAgents, Ttot, t0
        self.clock = self.history = None

    def reset(self, seed=None):
        "Reset the world and the schedule; returns each agent's first observation."
        self.world.reset(seed, self.nAgents)
        self.clock = Clock(self.nAgents, self.Ttot, self.t0)
        self.history = TeamHistory([()] * self.nAgents)
        return [self.world.observe(i) for i in range(self.nAgents)]

    def next_agent(self):
        "Agent whose next event is earliest (ties to the lower index), or None when all are done."
        return self.clock.peek()

    def observe(self, i):
        return self.world.observe(i)

    def step(self, action):
        "Apply the action for `next_agent()`; returns (reward, done)."
        i = self.clock.peek()
        if i is None:
            raise RuntimeError('all agents are done')
        t, _ = self.clock.pop()
        reward, dt, location, info = self.world.apply(i, action, t)
        self.history.add(Record(i, t, dt, 'step', location, reward, ()))
        return reward, not self.clock.push(t, dt, i)

## Tests

A toy world where agent `i`'s actions always take `durations[i]`, and members that repeat fixed actions.

In [ ]:
from pathlib import Path
import tempfile
from fastcore.test import test_eq, test_fail


class ToyWorld:
    def __init__(self, durations): self.durations, self.counts = list(durations), None
    def reset(self, seed, n): self.counts = [0] * n
    def observe(self, i): return self.counts[i]
    def apply(self, i, action, t=None):
        self.counts[i] += 1
        return float(action), self.durations[i], i, {}


class ToyMember(Member):
    def __init__(self, actions): self.actions, self.k, self.learnCalls = list(actions), 0, 0
    def act(self, obs):
        self.k += 1
        return self.actions[(self.k - 1) % len(self.actions)]
    def learn(self, obs, action, reward, info): self.learnCalls += 1

Order and ties (AC-3): durations 1.0, 1.5, 1.0 for 3 s give eight events.

In [ ]:
w = ToyWorld([1.0, 1.5, 1.0]); w.reset(0, 3)
ms = [ToyMember([1.0]) for _ in range(3)]
h = runTeam(ms, w, 3.0, t0=0.0)
test_eq([r.agent for r in h.records], [0, 1, 2, 0, 2, 1, 0, 2])
test_eq([r.t for r in h.records], [0.0, 0.0, 0.0, 1.0, 1.0, 1.5, 2.0, 2.0])
test_eq([len(h.forAgent(i)) for i in range(3)], [3, 2, 3])
test_eq([m.learnCalls for m in ms], [3, 2, 3])
test_eq([m.id for m in ms], [0, 1, 2])

w1 = ToyWorld([0.5]); w1.reset(0, 1)
h1 = runTeam([ToyMember([2.0])], w1, 2.0, t0=10.0)
test_eq([r.t for r in h1.records], [10.0, 10.5, 11.0, 11.5])
test_eq({r.reward for r in h1.records}, {2.0})

Errors (AC-4).

In [ ]:
test_fail(lambda: runTeam([], w, 1.0), contains='at least one')
bad = ToyMember([1.0]); bad.timeUnit = 'sample'
test_fail(lambda: runTeam([ToyMember([1.0]), bad], w, 1.0), contains='time unit')
w0 = ToyWorld([0.0]); w0.reset(0, 1)
test_fail(lambda: runTeam([ToyMember([1.0])], w0, 1.0), contains='> 0')

`TeamEnv` (AC-5): schedule, `done` flags, the end, reproducibility, and the same history as `runTeam` (FR-11).

In [ ]:
def drive(seed):
    env = TeamEnv(ToyWorld([1.0, 1.5, 1.0]), 3, 3.0)
    test_eq(env.reset(seed), [0, 0, 0])
    order, dones = [], []
    while (i := env.next_agent()) is not None:
        order.append(i)
        dones.append(env.step(float(i + 1))[1])
    return env, order, dones

env, order, dones = drive(0)
test_eq(order, [0, 1, 2, 0, 2, 1, 0, 2])
test_eq(dones, [False] * 5 + [True] * 3)
test_fail(lambda: env.step(1.0), contains='done')
test_eq(drive(0)[0].history.records, env.history.records)

wr = ToyWorld([1.0, 1.5, 1.0]); wr.reset(0, 3)
test_eq(runTeam([ToyMember([float(i + 1)]) for i in range(3)], wr, 3.0).records, env.history.records)

JSONL (AC-6) and seeds (AC-7).

In [ ]:
th = TeamHistory([('n', 'x', 'flag')])
th.add(Record(0, np.int64(3), 0.5, 'step', np.int64(7), np.float64(0.25), (np.int64(1), np.float64(2.5), np.bool_(True))))
th.add(Record(0, 3.5, 0.5, 'step', 8, -1.0, (2, 3.0, False)))
with tempfile.TemporaryDirectory() as d:
    rows = [json.loads(l) for l in th.toJSONL(Path(d) / 'sub' / 'h.jsonl').read_text().splitlines()]
test_eq(len(rows), 2)
test_eq(set(rows[0]), {'agent', 't', 'dt', 'kind', 'location', 'reward', 'extras'})
test_eq((rows[0]['t'], rows[0]['location'], rows[0]['reward']), (3, 7, 0.25))
test_eq(rows[0]['extras'], {'n': 1, 'x': 2.5, 'flag': True})
test_eq(rows[1]['extras'], {'n': 2, 'x': 3.0, 'flag': False})

test_eq([g.integers(1 << 30) for g in spawnRngs(0, 3)], [g.integers(1 << 30) for g in spawnRngs(0, 3)])
test_eq(len({g.random() for g in spawnRngs(0, 3)}), 3)
assert spawnRngs(1, 1)[0].random() != spawnRngs(0, 1)[0].random()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()